In [22]:
using Cosmology
using Jens
using Jens.LensModel.ComLens: CombinedLens
using Jens.LensModel: SIS, Shear
using Jens.LightModel: CompositeImage, ExtendedSource, PointImage, PointImages
using Jens.LensGenerator: LensedPlane, GenGrid
using Jens.LensSystem: LensTimeDelay
using Jens:LensGenerator as LG
using Jens.LensSolver: solve_images
using Jens.LensSystem: ForwardModel, render, LensFermat
using Jens.LensCosmo: time_delay_distance

# Quick look from LensedPlane

In [4]:
cosmo = Cosmology.FlatLCDM(0.7, 0.3, 0.0, 0.0)
z_lens, z_src = 0.3, 1.5

# modelling
cosmo = Cosmology.FlatLCDM(0.7, 0.3, 0.0, 0.0)
lens = CombinedLens(
    SIS   => (theta_E=1.0, xcentre=0.0, ycentre=0.0),
    Shear => (gamma1=0.05, gamma2=-0.02, xcentre=0.0, ycentre=0.0),
)
lp = LensedPlane(lens; z_lens=0.3, cosmology=cosmo)

# grid
grid = GenGrid(pix_n=256, pix_size=0.08)

# time delay
Delya_t = LensTimeDelay(xg, yg, [0.05, -0.03]; LensModel=lp, z_source=1.5);

# From LensSystem

In [24]:
const ASEC2_TO_RAD2 = (π / 180 / 3600)^2       # arcsec² → rad²
const C_LIGHT_MPC_S = 9.71561189025635e-15      # light speed (Mpc/s)

9.71561189025635e-15

In [26]:
agn = PointImage(flux=100.0, beta_x=0.05, beta_y=-0.03)  # 源位置
sys = ForwardModel(
    lens_plane   = LG.LensedPlane(lens; z_lens=0.3, cosmology=cosmo),
    source_plane = LG.LightPlane(agn; z=1.5),
    grid         = GenGrid(pix_n=256, pix_size=0.09),
)

images = solve_images(sys, 0.05, -0.03; z_source=1.5)


D_dt = time_delay_distance(cosmo, 0.3, 1.5)
delays = []
for (tx, ty, mu) in images
    τ  = LensFermat([tx;;], [ty;;], [0.05, -0.03];
                    LensModel=sys, z_source=1.5)[1,1]
    Deltat = D_dt * τ * ASEC2_TO_RAD2 / C_LIGHT_MPC_S
    push!(delays, Deltat)
end

Dleta_t_12 = abs(delays[1] - delays[2]) /  86400

8.57030390929899